# Figures 8--10: anatomy of the SXS:BBH:3617 $(4,4)$ multipole

Input: `data/fig08_10_sxs3617_N2_modes.npz`, containing extrapolation-N2 SXS:BBH:3617 modes with the peak shifted to $t=0$. The notebook recomputes direct weighted parent fits, the 6/10/15 QQNM sums, 440/441(/442) residual fits, and the $[t_0,t_0+15M_f]$ mismatch scan. It requires the separately released `tgr` package; no random seed is used.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'data').is_dir():
    raise RuntimeError('Run this notebook from the repository root.')
sys.path.insert(0, str(ROOT / 'src'))

import warnings, os
warnings.filterwarnings('ignore')
import numpy as np, matplotlib as mpl, matplotlib.pyplot as plt, lal
# PLOTTING OPTIONS (project-wide convention)
fig_width_pt = 3*246.0  # Get this from LaTeX using \showthe\columnwidth
inches_per_pt = 1.0/72.27               # Convert pt to inch
golden_mean = (np.sqrt(5)-1.0)/2.0         # Aesthetic ratio
fig_width = fig_width_pt*inches_per_pt  # width in inches
fig_height = fig_width*golden_mean      # height in inches
fig_size =  [fig_width,fig_height]
params = { 'axes.labelsize': 24,
          'font.family': 'serif',
          'font.serif': 'Computer Modern Raman',
          'font.size': 24,
          'legend.fontsize': 20,
          'xtick.labelsize': 24,
          'ytick.labelsize': 24,
          'axes.grid' : True,
          'text.usetex': True,
          'savefig.dpi' : 100,
          'lines.markersize' : 14,
          'figure.figsize': fig_size}
mpl.rcParams.update(params)
import anatomy44_tools as T
from anatomy44_tools import MODE22, QUAD15, QUAD10, QUAD6, QUAD3, QUAD1
print('tgr from', T.nr.__file__)

In [ ]:
W = {'SXS': T.load_sxs3617_npz(ROOT / 'data/fig08_10_sxs3617_N2_modes.npz')}
ANA = {}
for start in (4.0, 5.0):
    waveform = W['SXS']
    ANA[('SXS', start, 'direct')] = T.anatomy(
        waveform, start, T.parents_direct(waveform, start))

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
w = W['SXS']; mf = w.Mf
for k in [4.0]:
    r = ANA[('SXS', k, 'direct')]
    h44, ser, sums = r['h44'], r['series'], r['sums']
    fig, ax = plt.subplots(figsize=(12, 8))
    tt = h44.sample_times.numpy() / mf
    ax.plot(tt, h44.real().numpy() / mf, label='SXS:BBH:3617 (4,4)', color='black', lw=3)
    for m in QUAD6:
        lw, alpha, ls = (2, 1, '--') if m == '220220' else (1, 0.5, '-')
        ax.plot(tt, ser[m].real().numpy() / mf, alpha=alpha, lw=lw, ls=ls, label=f'${m[:3]}\\times {m[3:]}$')
    ax.plot(tt, sums[6].real().numpy() / mf, label='Sum of 6 QQNMs', ls='-.', lw=2, color='blue')
    ax.plot(tt, sums[10].real().numpy() / mf, label='Sum of 10 QQNMs', ls='--', lw=2, color='red')
    ax.plot(tt, sums[15].real().numpy() / mf, label='Sum of 15 QQNMs', ls='--', lw=2, color='purple')
    ax.set_xlim(k - 0.5, 20); ax.set_ylim(-0.2, 0.2)
    ax.set_xlabel('Time [$M_\\mathrm{f}$]'); ax.set_ylabel('$\\mathrm{Re}(rh)$ [$M_\\mathrm{f}$]')
    ax.legend(ncols=2, fontsize=13)
    ins = inset_axes(ax, width='50%', height='30%', loc='lower right', borderpad=1.0)
    ins.plot(tt, h44.real().numpy() / mf, color='black', lw=2)
    ins.plot(tt, r['lin_only'].real().numpy() / mf, label=f"linear fit (mismatch {r['mm_lin_only']:.2f})", ls=':', lw=2, color='green')
    ins.plot(tt, r['model'].real().numpy() / mf, label=f"10 QQNMs + linear (mismatch {r['mm_model']:.3f})", ls='--', lw=2, color='orange')
    ins.set_xlim(k, 40); ins.set_ylim(-0.06, 0.06); ins.legend(fontsize=11, ncol=1); ins.grid(False); ins.tick_params(labelsize=12)
    fig.savefig(ROOT / f'figures/qqnm_{k:.0f}Mf.pdf', bbox_inches='tight')
    plt.show()

In [ ]:
LIN_STARTS = [4., 5., 6., 7., 8., 9., 10.]

w = W['SXS']
def multistart_lin44(w, kind, method, lin_modes=('440', '441'), omitted=('442',)):
    """fit lin_modes at each start of LIN_STARTS on kind='resid10' (h44 - Sum10, direct parents at that start) or 'raw' (h44)."""
    out = {}
    for k in LIN_STARTS:
        h44 = T.h44_from(w, k); target = h44
        if kind == 'resid10':
            Q = T.qqnm_amps(w, T.parents_direct(w, k)); target = h44 - T.sum_series(T.qqnm_series(w, k, Q), QUAD10)
        A, _ = T.fit_linear44(w, target, h44, lin_modes, method, omitted)
        out[k] = dict(A=A, t0=float(h44.start_time))
    return out
def a_at_ref(w, fits, m, ref):
    ltab = T.lin_table(w, [m])
    return np.array([fits[k]['A'][m] * np.exp(-1j * ltab.omega(m) * (w.t(ref) - fits[k]['t0'])) for k in LIN_STARTS])
MS44 = {(kind, meth): multistart_lin44(w, kind, meth) for kind in ['resid10', 'raw'] for meth in ['wls', 'ols']}
MS44[('resid10', 'wls+442')] = multistart_lin44(w, 'resid10', 'wls', ('440', '441', '442'), ('443',))
for ref in [4.0, 5.0]:
    print(f"SXS |A|/Mf propagated to {ref:.0f} Mf, per fit start ({', '.join(f'{k:.0f}' for k in LIN_STARTS)} Mf):")
    for key, fits in MS44.items():
        for m in (['440', '441', '442'] if '442' in key[1] else ['440', '441']):
            a = a_at_ref(w, fits, m, ref)
            print(f"   {key[0]:>8} {key[1].upper():>7} {m}: " + ' '.join(f"{v:7.4f}" for v in np.abs(a) * w.conv)
                  + f"   std/mean = {np.std(np.abs(a))/np.mean(np.abs(a)):.4f}")
    print()

REF = 4.0; EXT = np.arange(4.0, 10.25, 0.25)
panels = [(('resid10', 'wls'), ['440', '441']),
          (('resid10', 'wls+442'), ['440', '441', '442'])]
fig, axes = plt.subplots(3, 2, figsize=(1.4*fig_width, 2.1*fig_height), sharex=True)
for col, (key, modes) in enumerate(panels):
    fits = MS44[key]
    for row, m in enumerate(modes):
        ax = axes[row, col]; ltab = T.lin_table(w, [m])
        for i, k in enumerate(LIN_STARTS):
            c = f'C{i}'; A = fits[k]['A'][m]; t0 = fits[k]['t0']
            ax.scatter(k, abs(A) * w.conv, s=80, c=c, zorder=3)
            ax.plot(EXT, np.abs(A * np.exp(-1j * ltab.omega(m) * (EXT * w.Mf - t0))) * w.conv, c=c, lw=1.5, zorder=1)
        a = a_at_ref(w, fits, m, REF)
        ax.set_yscale('log'); ax.set_ylabel(f'$r|A_{{{m}}}|/M_\\mathrm{{f}}$')
        ax.text(0.05, 0.08, f'std/mean = {np.std(np.abs(a))/np.mean(np.abs(a)):.4f}',
                transform=ax.transAxes, fontsize=13,
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        ax.grid(True, alpha=0.3)
axes[2, 0].axis('off'); axes[1, 0].tick_params(labelbottom=True)
axes[1, 0].set_xlabel('time after merger [$M_\\mathrm{f}$]'); axes[2, 1].set_xlabel('time after merger [$M_\\mathrm{f}$]')
plt.subplots_adjust(hspace=0.12, wspace=0.35)
fig.savefig(ROOT / 'figures/NR10q_440441_ampconv_wls_4Mf.pdf', bbox_inches='tight')
plt.show()

In [ ]:
w = W['SXS']
DELTAS = np.linspace(0.0, 1.5, 31)
DSCAN = {}
for k in [4.0, 5.0]:
    r = ANA[('SXS', k, 'direct')]
    h44, s10 = r['h44'], r['sums'][10]
    mm, a440, a441 = [], [], []
    for d in DELTAS:
        A, lin = T.fit_linear44(w, h44 - float(d) * s10, h44, ('440', '441'), 'ols', ('442',))
        # mismatch over [t0, t0+15 Mf] only (matches the committed paper figure
        # and the App. D text; the tail beyond 15 Mf is numerically contaminated)
        n15 = int(round(float(w.t(k + 15.0) - w.t(k)) * h44.sample_rate))
        model = (lin + float(d) * s10).numpy()[:n15]
        mm.append(T.mismatch(model, h44.numpy()[:n15])); a440.append(abs(A['440']) * w.conv); a441.append(abs(A['441']) * w.conv)
    DSCAN[k] = dict(mm=np.array(mm), a440=np.array(a440), a441=np.array(a441))
    i1, im = np.argmin(abs(DELTAS - 1.0)), int(np.argmin(mm))
    print(f"{k:.0f} Mf: mismatch(lin + d*S10, h44): d=0 {mm[0]:.4f}, d=1 {mm[i1]:.4f}, min {mm[im]:.4f} at d={DELTAS[im]:.2f};  "
          f"|A440|/Mf {a440[0]:.4f} -> {a440[i1]:.4f}, |A441|/Mf {a441[0]:.4f} -> {a441[i1]:.4f}  (d=0 -> 1)")
    _, lin6 = T.fit_linear44(w, h44 - r['sums'][6], h44)
    print(f"      mismatch(S6, S10) = {T.mismatch(r['sums'][6], s10):.4f};  6 QQNM + linear fit: {T.mismatch(lin6 + r['sums'][6], h44):.4f} "
          f"(10 QQNM + linear: {r['mm_model']:.4f});  sqrt(E_S10/E_S6) - 1 = {np.sqrt(T.energy(s10)/T.energy(r['sums'][6]))-1:+.1%}")
fig, ax = plt.subplots(figsize=(10, 5.5))
for k, c in [(4.0, 'C1'), (5.0, 'C0')]:
    ax.semilogy(DELTAS, DSCAN[k]['mm'], '-', color=c, lw=1.5, label=f'start time ${k:.0f}\\,M_\\mathrm{{f}}$')
ax.axvline(1.0, color='gray', ls=':', lw=1)
ax.set_ylabel(r'mismatch$(\mathrm{linear\ fit} + \delta\,\Sigma_{10},\ h_{44})$'); ax.set_xlabel(r'$\delta$ (subtracted QQNM fraction)')
ax.legend(); ax.set_xlim(1.52, -0.02)
fig.savefig(ROOT / 'figures/NR10q_linfit_vs_delta_4Mf.pdf', bbox_inches='tight')
plt.show()